## Data analysis

In [68]:
import sys
import os

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [69]:
import pandas as pd
import numpy as np
import json
from ecv_rank import *
from src.solvers.weighted_network import ASE
from src.methods import * 
import matplotlib.pyplot as plt
import requests
from scipy.sparse.linalg import eigsh
import pickle

import threading

In [70]:
migration_flow = pd.read_csv('data_cleaned/county_migration_cleaned.csv')
gdp_change_2012_2016 = pd.read_csv('data_cleaned/gdp_change_2012_2016.csv')
gdp_per_capita = pd.read_csv('data_cleaned/county_gdp_per_capita_2012_2020.csv')
voting_data = pd.read_csv('data_cleaned/county_voting_cleaned.csv')
voting_change = pd.read_csv('data_cleaned/votes_change.csv')

In [71]:
gdp_change_2012_2016.dropna(inplace=True)

gdp_change_2012_2016['county_fips'] = gdp_change_2012_2016['county_fips'].astype(str).str.zfill(6)
gdp_change_2012_2016['county_fips'] = gdp_change_2012_2016['county_fips'].apply(lambda x: x[0:3]+'_'+x[3:6])
gdp_change_2012_2016.head(3)

,county_fips,2012,2016,gdp_pc_change_2012_2016
0,001_001,31068.071400,32402.393864,1334.322464
1,001_003,31397.947166,33848.416354,2450.469188
2,001_005,27374.015604,29433.666845,2059.651241


In [72]:
gdp_per_capita.dropna(inplace=True)

gdp_per_capita['county_fips'] = gdp_per_capita['county_fips'].astype(str).str.zfill(6)
gdp_per_capita['county_fips'] = gdp_per_capita['county_fips'].apply(lambda x: x[0:3]+'_'+x[3:6])
gdp_per_capita.head(3)

,county_fips,county_name,year,real_gdp_thousands,population,real_gdp,real_gdp_per_capita,state_code,county_code
0,001_001,"Autauga, AL",2012,1726577,55574,1726577000,31068.071400,1,1
1,001_001,"Autauga, AL",2013,1618151,55619,1618151000,29093.493231,1,1
2,001_001,"Autauga, AL",2014,1629762,56062,1629762000,29070.707431,1,1


In [73]:
voting_data.dropna(inplace=True)

voting_data['county_fips'] = voting_data['county_fips'].astype(int).astype(str).str.zfill(6)
voting_data['county_fips'] = voting_data['county_fips'].apply(lambda x: x[0:3]+'_'+x[3:6])
voting_data.head(3)

,state_name,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode,county_code,state_code
0,alabama,autauga,2012,AL,001_001,US PRESIDENT,BARACK OBAMA,DEMOCRAT,6363.0,23932,20260225,TOTAL,1,1
1,alabama,autauga,2012,AL,001_001,US PRESIDENT,MITT ROMNEY,REPUBLICAN,17379.0,23932,20260225,TOTAL,1,1
2,alabama,autauga,2012,AL,001_001,US PRESIDENT,OTHER,OTHER,190.0,23932,20260225,TOTAL,1,1


In [74]:
voting_change.dropna(inplace=True)

voting_change['state_code'] = voting_change['state_code'].astype(int).astype(str).str.zfill(3)
voting_change['county_code'] = voting_change['county_code'].astype(int).astype(str).str.zfill(3)
voting_change['county_fips'] = voting_change['state_code'].astype(str) + '_' + voting_change['county_code']

voting_change.head(3)

,state_code,county_code,dem_change_2012_2016,dem_change_2016_2020,rep_change_2012_2016,rep_change_2016_2020,county_fips
1,001,001,-427.0,1567.0,793.0,1666.0,001_001
2,001,003,34.0,6120.0,6867.0,10661.0,001_003
3,001,005,-1041.0,-55.0,-96.0,168.0,001_005


## Dataset build

In [75]:
migration_flow['state_code'] = migration_flow['state_code'].astype(str).str.zfill(3)
migration_flow['county_code'] = migration_flow['county_code'].astype(str).str.zfill(3)
migration_flow['state_code_B'] = migration_flow['state_code_B'].astype(str).str.zfill(3)
migration_flow['county_code_B'] = migration_flow['county_code_B'].astype(str).str.zfill(3)

In [76]:
migration_flow['code_A'] = migration_flow['state_code'].astype(str) + '_' + migration_flow['county_code']
migration_flow['code_B'] = migration_flow['state_code_B'].astype(str) + '_' + migration_flow['county_code_B']

In [77]:
# ============================================================
# Create county code <-> county name mappings
# ============================================================

df_names = (
    migration_flow
    .groupby('code_A')[['state_name', 'county_name']]
    .first()
    .rename(columns={
        'state_name': 'State Name',
        'county_name': 'County Name'
    })
)


codetoname = {
    code: row['State Name'] + '_' + row['County Name'].replace(' ', '_')
    for code, row in df_names.iterrows()
}

nametocode = {
    name: code
    for code, name in codetoname.items()
}


# ============================================================
# Create index mappings
# ============================================================

nametoidx = {
    name: i
    for i, name in enumerate(nametocode.keys())
}

idxtoname = {
    i: name
    for name, i in nametoidx.items()
}


codetoidx = {
    code: i
    for i, code in enumerate(codetoname.keys())
}

idxtocode = {
    i: code
    for code, i in codetoidx.items()
}

In [78]:
# ============================================================
# Create adjacency matrix
# ============================================================

n = migration_flow.groupby('state_name')['county_code'].nunique().sum()

adj_matrix = np.zeros((n, n))

interactions = 0


for _, row in migration_flow.iterrows():

    i = codetoidx[row['code_A']]
    j = codetoidx[row['code_B']]

    adj_matrix[j, i] = row['gross_migration']

    interactions += 1

In [79]:
# ============================================================
# Build Laplacian
# ============================================================

d = adj_matrix.sum(axis=1)

eps = 1e-10

dinv = 1 / np.sqrt(d + eps)

L = dinv[:, None] * adj_matrix * dinv[None, :]

In [80]:
# def ecv_computation():
#     result = ecv_rank(
#         L,
#         max_k=200,
#         B=10,
#         holdout_p=0.1,
#         weighted=True,
#         mode="undirected",
#         random_state=2,
#     )
#     with open('ecv_res_laplacian.pkl', 'wb') as f:
#         pickle.dump(result, f)

# thread = threading.Thread(target=ecv_computation)
# thread.start()

In [81]:
# print(thread.is_alive())

In [82]:
from pandas import read_pickle

with open('ecv_res.pkl', 'rb') as f:
    res = read_pickle(f)
    
with open('ecv_res_laplacian.pkl', 'rb') as f:
    res_laplacian = read_pickle(f)

## Prepare vectors

In [83]:
voting_change_vector = np.ones(adj_matrix.shape[0]) * np.inf

used_codes = set()

for i, row in voting_change.iterrows():
    idx = codetoidx[row['county_fips']]
    voting_change_vector[idx] = row['rep_change_2012_2016']
    used_codes.add(row['county_fips'])

assert used_codes == set(codetoidx.keys()), "Not all county codes are used in voting_change"

KeyError: '051_003'

In [84]:
voting_change

,state_code,county_code,dem_change_2012_2016,dem_change_2016_2020,rep_change_2012_2016,rep_change_2016_2020,county_fips
1,001,001,-427.0,1567.0,793.0,1666.0,001_001
2,001,003,34.0,6120.0,6867.0,10661.0,001_003
3,001,005,-1041.0,-55.0,-96.0,168.0,001_005
4,001,007,-328.0,112.0,606.0,787.0,001_007
5,001,009,-814.0,484.0,2102.0,1852.0,001_009
...,...,...,...,...,...,...,...
3104,056,037,-1543.0,592.0,726.0,75.0,056_037
3105,056,039,1101.0,2534.0,-937.0,420.0,056_039
3106,056,041,-426.0,389.0,-461.0,1342.0,056_041
3107,056,043,-262.0,119.0,-103.0,334.0,056_043


In [ ]:
gdp_change_vector = np.ones(adj_matrix.shape[0]) * np.inf

used_codes = set()

for i, row in gdp_change_2012_2016.iterrows():
    idx = codetoidx[row['county_fips']]
    gdp_change_vector[idx] = row['gdp_pc_change_2012_2016']
    used_codes.add(row['county_fips'])

assert used_codes == set(codetoidx.keys()), "Not all county codes are used in gdp_change_2012_2016"

AssertionError: Not all county codes are used in gdp_change_2012_2016

In [ ]:
set(codetoidx.keys()).difference(set(used_codes))

{'051_003',
 '051_015',
 '051_053',
 '051_059',
 '051_089',
 '051_121',
 '051_149',
 '051_153',
 '051_161',
 '051_520',
 '051_530',
 '051_580',
 '051_590',
 '051_595',
 '051_600',
 '051_620',
 '051_630',
 '051_640',
 '051_660',
 '051_678',
 '051_680',
 '051_683',
 '051_720',
 '051_730',
 '051_735',
 '051_820',
 '051_830',
 '051_840'}

In [ ]:
gdp_change_2012_2016['state_code'] = gdp_change_2012_2016['county_fips'].apply(lambda x: x.split('_')[0])
gdp_change_2012_2016['state_code'].unique()

array(['001', '004', '005', '006', '008', '009', '010', '011', '012',
       '013', '016', '017', '018', '019', '020', '021', '022', '023',
       '024', '025', '026', '027', '028', '029', '030', '031', '032',
       '033', '034', '035', '036', '037', '038', '039', '040', '041',
       '042', '044', '045', '046', '047', '048', '049', '050', '051',
       '053', '054', '055', '056'], dtype=object)

In [ ]:
voting_change[voting_change['state_code'] == '015']

,state_code,county_code,dem_change_2012_2016,dem_change_2016_2020,rep_change_2012_2016,rep_change_2016_2020,county_fips
518,015,001,-5965.0,17472.0,2748.0,9396.0,015_001
519,015,003,-28653.0,63170.0,1865.0,45924.0,015_003
520,015,007,-2185.0,4769.0,1453.0,4008.0,015_007
521,015,009,-2572.0,13825.0,1844.0,8680.0,015_009


In [ ]:
gdp_change_2012_2016[gdp_change_2012_2016['county_fips']=='051_913']

,county_fips,2012,2016,gdp_pc_change_2012_2016


In [ ]:
for x in range(900, 1000):
    if len(gdp_per_capita[gdp_per_capita['county_fips']==f'015_{x}']) > 0:
        print(gdp_per_capita[gdp_per_capita['county_fips']==f'015_{x}'].iloc[1][['county_name', 'county_fips']])

county_name    Maui + Kalawao, HI*
county_fips                015_901
Name: 4762, dtype: object


In [ ]:
for x in range(900, 1000):
    if len(voting_change[voting_change['county_fips']==f'015_{x}']) > 0:
        print(voting_change[voting_change['county_fips']==f'015_{x}'].iloc[1][['county_name', 'county_fips']])

,county_fips,county_name,year,real_gdp_thousands,population,real_gdp,real_gdp_per_capita,state_code,county_code
25947,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2012,107372818,1156454,107372818000,92846.596579,51,919
25948,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2013,108458197,1168500,108458197000,92818.311510,51,919
25949,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2014,108533102,1173144,108533102000,92514.731354,51,919
25950,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2015,109923813,1177529,109923813000,93351.257591,51,919
25951,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2016,111444480,1181273,111444480000,94342.696396,51,919
25952,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2017,114188029,1186154,114188029000,96267.456839,51,919
25953,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2018,119273660,1186544,119273660000,100521.902264,51,919
25954,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2019,125628136,1189262,125628136000,105635.373871,51,919
25955,051_919,"Fairfax, Fairfax City + Falls Church, VA*",2020,125622450,1187536,125622450000,105784.119387,51,919


In [ ]:
voting_change

,state_code,county_code,dem_change_2012_2016,dem_change_2016_2020,rep_change_2012_2016,rep_change_2016_2020,county_fips
1,001,001,-427.0,1567.0,793.0,1666.0,001_001
2,001,003,34.0,6120.0,6867.0,10661.0,001_003
3,001,005,-1041.0,-55.0,-96.0,168.0,001_005
4,001,007,-328.0,112.0,606.0,787.0,001_007
5,001,009,-814.0,484.0,2102.0,1852.0,001_009
...,...,...,...,...,...,...,...
3108,056,037,-1543.0,592.0,726.0,75.0,056_037
3109,056,039,1101.0,2534.0,-937.0,420.0,056_039
3110,056,041,-426.0,389.0,-461.0,1342.0,056_041
3111,056,043,-262.0,119.0,-103.0,334.0,056_043


In [ ]:
gdp_change_2012_2016['state_code'] = gdp_change_2012_2016['county_fips'].apply(lambda x: x.split('_')[0])
gdp_change_2012_2016['county_code'] = gdp_change_2012_2016['county_fips'].apply(lambda x: x.split('_')[1])

gdp_change_2012_2016[gdp_change_2012_2016['state_code'] == '015']

,county_fips,2012,2016,gdp_pc_change_2012_2016,state_code,county_code
526,015_001,38202.623389,40545.722265,2343.098876,015,001
527,015_003,60393.373776,62897.178187,2503.804411,015,003
528,015_007,48256.564163,52002.389866,3745.825703,015,007
529,015_901,53101.807339,58201.274539,5099.467200,015,901


In [ ]:
set(voting_change['county_fips'].values) == set(codetoidx.keys())

True

In [ ]:
def test_computation():
    test = MultivariateACTest(k=19, solver=ASE)
    data = {
        'A': adj_matrix,
        'Y': voting_change_vector.reshape(-1, 1)
    }

    test.fit(data, M=1, n_permutations=1000, random_state=1, 
            permutation_type='covariates', alpha=0.05, verbose=True)
    
    with open('test_res.pkl', 'wb') as f:
        pickle.dump(test.get_estimated(), f)

thread = threading.Thread(target=test_computation)
thread.start()

In [ ]:
with open('test_res.pkl', 'rb') as f:
    test = read_pickle(f)

test['p-value']

0.009900990099009901